# Sampling the DDXPlus patient data

Draw smaller, reproducible subsets of the patient splits for exploration and quick experiments:

1. Simple random sample
2. Stratified sample (equal number of patients per pathology, or proportional)
3. Filtered sample (by sex, age range, pathology)
4. Read a sampled patient in plain English
5. Save a sample to `data/samples/` (git-ignored)

Change the settings in the next cell and re-run.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():  # notebook is opened from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_conditions, load_evidences, load_patients

pd.set_option("display.max_colwidth", 120)

# --- settings ---
SPLIT = "train"     # "train", "validate" or "test"
NROWS = None        # e.g. 200_000 to load only the first rows (faster, but not a random subset)
SEED = 42           # fixed seed so samples are reproducible

## Load

In [ ]:
patients = load_patients(SPLIT, nrows=NROWS)
evidences = load_evidences()
conditions = load_conditions()

print(f"{SPLIT}: {len(patients):,} patients, {patients.PATHOLOGY.nunique()} pathologies")
patients.head()

In [5]:
# How unbalanced are the pathologies? This matters for choosing a sampling method.
counts = patients.PATHOLOGY.value_counts()
print(f"Largest class:  {counts.index[0]} ({counts.iloc[0]:,})")
print(f"Smallest class: {counts.index[-1]} ({counts.iloc[-1]:,})")
counts.to_frame("patients").assign(share=lambda d: (d.patients / d.patients.sum()).round(4))

Largest class:  URTI (64,368)
Smallest class: Bronchiolitis (261)


,patients,share
PATHOLOGY,,
URTI,64368,0.0628
Viral pharyngitis,61642,0.0601
Anemia,50665,0.0494
HIV (initial infection),29013,0.0283
Localized edema,27825,0.0271
Anaphylaxis,27718,0.0270
Pulmonary embolism,27468,0.0268
Influenza,26812,0.0261
Bronchitis,26400,0.0257


## 1. Simple random sample

Every patient has the same chance of being picked, so the pathology mix follows the full split.

In [6]:
N = 1_000
random_sample = patients.sample(n=N, random_state=SEED)
random_sample.PATHOLOGY.value_counts().head(10)

PATHOLOGY
Viral pharyngitis           82
URTI                        67
Anemia                      38
Allergic sinusitis          35
Influenza                   32
Acute dystonic reactions    31
Localized edema             29
Anaphylaxis                 28
Acute otitis media          26
GERD                        25
Name: count, dtype: int64

## 2. Stratified sample

- **Balanced:** the same number of patients per pathology (capped at the class size), useful when rare pathologies would otherwise barely appear.
- **Proportional:** the same fraction from every pathology, which keeps the original class mix exactly.

In [7]:
def sample_per_class(df, n_per_class, by="PATHOLOGY", seed=SEED):
    """Up to n_per_class rows from each group of `by`."""
    shuffled = df.sample(frac=1, random_state=seed)
    return shuffled.groupby(by, observed=True).head(n_per_class).reset_index(drop=True)


def sample_proportional(df, frac, by="PATHOLOGY", seed=SEED):
    """The same fraction of rows from each group of `by`."""
    return (
        df.groupby(by, observed=True, group_keys=False)
          .sample(frac=frac, random_state=seed)
          .reset_index(drop=True)
    )


balanced = sample_per_class(patients, n_per_class=100)
proportional = sample_proportional(patients, frac=0.01)

print(f"balanced:     {len(balanced):,} rows, {balanced.PATHOLOGY.value_counts().min()}-"
      f"{balanced.PATHOLOGY.value_counts().max()} per pathology")
print(f"proportional: {len(proportional):,} rows")

balanced:     4,900 rows, 100-100 per pathology
proportional: 10,259 rows


In [9]:
# Compare class shares: full split vs. the two stratified samples
pd.DataFrame({
    "full": patients.PATHOLOGY.value_counts(normalize=True),
    "proportional": proportional.PATHOLOGY.value_counts(normalize=True),
    "balanced": balanced.PATHOLOGY.value_counts(normalize=True),
}).round(4).head(10)

,full,proportional,balanced
PATHOLOGY,,,
Acute COPD exacerbation / infection,0.0172,0.0173,0.0204
Acute dystonic reactions,0.0253,0.0253,0.0204
Acute laryngitis,0.0235,0.0235,0.0204
Acute otitis media,0.0253,0.0252,0.0204
Acute pulmonary edema,0.0185,0.0185,0.0204
Acute rhinosinusitis,0.0132,0.0133,0.0204
Allergic sinusitis,0.0255,0.0255,0.0204
Anaphylaxis,0.0270,0.0270,0.0204
Anemia,0.0494,0.0494,0.0204


## 3. Filtered sample

Pick a subgroup first, then sample inside it. Leave a filter as `None` to skip it.

In [10]:
def sample_filtered(df, n, sex=None, age_min=None, age_max=None, pathologies=None, seed=SEED):
    mask = pd.Series(True, index=df.index)
    if sex is not None:
        mask &= df.SEX == sex
    if age_min is not None:
        mask &= df.AGE >= age_min
    if age_max is not None:
        mask &= df.AGE <= age_max
    if pathologies is not None:
        mask &= df.PATHOLOGY.isin(pathologies)
    subset = df[mask]
    print(f"{len(subset):,} patients match the filters")
    return subset.sample(n=min(n, len(subset)), random_state=seed)


filtered = sample_filtered(patients, n=500, sex="F", age_min=60, pathologies=["Pneumonia", "Bronchitis", "URTI"])
filtered.head()

11,451 patients match the filters


,AGE,DIFFERENTIAL_DIAGNOSIS,SEX,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
556370,65,"[(Pneumonia, 0.139902), (Bronchitis, 0.1311), (Tuberculosis, 0.11164), (Influenza, 0.109742), (Pulmonary neoplasm, 0...",F,Pneumonia,"[E_45, E_53, E_54_@_V_192, E_55_@_V_55, E_55_@_V_56, E_55_@_V_170, E_56_@_7, E_57_@_V_123, E_58_@_0, E_59_@_1, E_77,...",E_53
444993,66,"[(Acute COPD exacerbation / infection, 0.071208), (Bronchitis, 0.065913), (Pneumonia, 0.064729), (Bronchiectasis, 0....",F,Bronchitis,"[E_53, E_54_@_V_161, E_54_@_V_181, E_55_@_V_29, E_55_@_V_56, E_55_@_V_101, E_55_@_V_160, E_55_@_V_171, E_56_@_4, E_5...",E_77
807658,60,"[(Pneumonia, 0.07697), (Pulmonary neoplasm, 0.0748), (Bronchitis, 0.070012), (Bronchiectasis, 0.067323), (Tuberculos...",F,Pneumonia,"[E_45, E_53, E_54_@_V_192, E_55_@_V_55, E_55_@_V_56, E_55_@_V_170, E_56_@_1, E_57_@_V_123, E_58_@_7, E_59_@_1, E_66,...",E_129
172695,66,"[(URTI, 0.180451), (Bronchitis, 0.170117), (Cluster headache, 0.137863), (Bronchiectasis, 0.115495), (Chronic rhinos...",F,URTI,"[E_48, E_50, E_53, E_54_@_V_161, E_54_@_V_183, E_55_@_V_62, E_55_@_V_89, E_55_@_V_108, E_55_@_V_124, E_56_@_5, E_57_...",E_97
431261,62,"[(URTI, 0.212674), (Influenza, 0.193334), (Bronchitis, 0.187186), (Tuberculosis, 0.139111), (Pneumonia, 0.123387), (...",F,URTI,"[E_48, E_53, E_54_@_V_154, E_55_@_V_62, E_55_@_V_89, E_55_@_V_108, E_55_@_V_124, E_55_@_V_166, E_56_@_6, E_57_@_V_12...",E_53


## 4. Read a patient in plain English

Each entry in `EVIDENCES` is either a yes/no evidence code (`E_91`) or a code with a value
(`E_54_@_V_112`, `E_56_@_6`). This turns them into the English question and answer.

In [11]:
def decode_evidence(item):
    code, _, value = item.partition("_@_")
    ev = evidences.loc[code]
    if not value:
        return ev.question_en, "yes"
    answer = ev.value_meaning_en.get(value, value)
    return ev.question_en, {"N": "no", "Y": "yes"}.get(answer, answer)  # dataset uses N/Y for some answers


def describe_patient(row):
    print(f"Age {row.AGE}, sex {row.SEX}")
    print(f"Pathology: {row.PATHOLOGY} (ICD-10 {conditions.loc[row.PATHOLOGY, 'icd10_id']})")
    print(f"First complaint: {evidences.loc[row.INITIAL_EVIDENCE, 'question_en']}")
    decoded = pd.DataFrame([decode_evidence(e) for e in row.EVIDENCES], columns=["question", "answer"])
    # multi-choice questions appear once per chosen answer; join them
    decoded = decoded.groupby("question", sort=False).answer.agg(", ".join).reset_index()
    top = ", ".join(f"{p} ({prob:.2f})" for p, prob in row.DIFFERENTIAL_DIAGNOSIS[:3])
    print(f"Top differential: {top}")
    return decoded


describe_patient(random_sample.iloc[0])

Age 49, sex M
Pathology: Allergic sinusitis (ICD-10 J30)
First complaint: Is your nose or the back of your throat itchy?
Top differential: Allergic sinusitis (0.41), Bronchitis (0.34), URTI (0.25)


,question,answer
0,"Do you have any close family members who suffer from allergies (any type), hay fever or eczema?",yes
1,Do you have any family members who have asthma?,yes
2,Do you have asthma or have you ever had to use a bronchodilator in the past?,yes
3,Is your nose or the back of your throat itchy?,yes
4,Do you have nasal congestion or a clear runny nose?,yes
5,Do you have a cough?,yes
6,Have you traveled out of the country in the last 4 weeks?,no
7,Do you live in in a big city?,yes


## 5. Save a sample

In [12]:
OUT_DIR = ROOT / "data" / "samples"   # data/ is git-ignored
OUT_DIR.mkdir(parents=True, exist_ok=True)

to_save = balanced   # pick any sample from above
out_path = OUT_DIR / f"{SPLIT}_balanced_seed{SEED}.pkl"
to_save.to_pickle(out_path)   # pickle keeps the list columns and categories intact
print(f"Saved {len(to_save):,} rows to {out_path.relative_to(ROOT)}")

# Load it back later with: pd.read_pickle(out_path)

Saved 4,900 rows to data/samples/train_balanced_seed42.pkl
